# Post-Train VAE From Saved ChemBL+Zinc Checkpoint

This notebook shows how to resume from the saved pretrained VAE and post-train on tox21.

Use case: handoff to a colleague who has not done post-training before.

## What this notebook does
1. Loads `paper_like_selfies_chembl_zinc.pt` from `artifacts/pretraining_checkpoints/`.
2. Rebuilds the same VAE architecture and tokenizer mapping used during pretraining.
3. Loads tox21 train/val/test data, converts to SELFIES, and encodes with pretrained vocab.
4. Runs optional post-training (fine-tuning) and saves a new post-trained checkpoint.

## Important
- You do not need to retrain from scratch.
- Post-training starts from pretrained weights loaded from checkpoint.
- Unknown tox21 tokens are mapped to `<UNK>` so encoding stays compatible.


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import selfies as sf
from tqdm.auto import tqdm

SEED = 42
POST_EPOCHS = 10
BATCH_SIZE = 128
POST_LR = 3e-4
EVAL_EVERY = 1

# Toggle this to start/skip training when running all cells.
RUN_POST_TRAIN = False

DATA_ROOT = Path("data")
TOX21_TRAIN_PATH = DATA_ROOT / "Train" / "tox21_train_clean.csv"
TOX21_VAL_PATH = DATA_ROOT / "Val" / "tox21_val_clean.csv"
TOX21_TEST_PATH = DATA_ROOT / "Test" / "tox21_test_clean.csv"

PRETRAIN_CKPT = Path("artifacts") / "pretraining_checkpoints" / "paper_like_selfies_chembl_zinc.pt"
POSTTRAIN_SAVE_DIR = Path("artifacts") / "posttraining_checkpoints"
POSTTRAIN_SAVE_DIR.mkdir(parents=True, exist_ok=True)
POSTTRAIN_CKPT = POSTTRAIN_SAVE_DIR / "paper_like_selfies_chembl_zinc_posttrained_tox21.pt"

for p in [TOX21_TRAIN_PATH, TOX21_VAL_PATH, TOX21_TEST_PATH, PRETRAIN_CKPT]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print("device:", device)
print("torch:", torch.__version__)
print("selfies:", sf.__version__)


## Load Pretrained Checkpoint
This is the key resume step: we recover model weights and tokenizer metadata from pretraining.


In [ ]:
ckpt = torch.load(PRETRAIN_CKPT, map_location="cpu")
print("checkpoint keys:", sorted(ckpt.keys()))

token_to_idx = ckpt["token_to_idx"]
idx_to_token = {i: tok for tok, i in token_to_idx.items()}

PAD = "<PAD>"
UNK = "<UNK>"
EOS = "<EOS>"
PAD_IDX = token_to_idx[PAD]
UNK_IDX = token_to_idx[UNK]
EOS_IDX = token_to_idx[EOS]

SEQ_LEN = int(ckpt["seq_len"])
VOCAB_SIZE = int(ckpt["vocab_size"])
MAX_LEN = int(ckpt["max_len"])

# Infer latent dim directly from saved weights for safety.
LATENT_DIM = int(ckpt["model_state_dict"]["linear_1.weight"].shape[0])

print(f"SEQ_LEN={SEQ_LEN}, VOCAB_SIZE={VOCAB_SIZE}, MAX_LEN={MAX_LEN}, LATENT_DIM={LATENT_DIM}")
print(f"Tokenizer size from checkpoint: {len(token_to_idx)}")


## Load tox21 Data And Convert To SELFIES
Tox21 files are already split into train/val/test in this project.


In [ ]:
def load_smiles(path: Path) -> list[str]:
    df = pd.read_csv(path)
    if "canonical_smiles" not in df.columns:
        raise ValueError(f"{path} does not contain canonical_smiles")
    smiles = df["canonical_smiles"].dropna().astype(str).tolist()
    return list(dict.fromkeys(smiles))


def smiles_to_selfies(smiles_list: list[str]) -> tuple[list[str], int]:
    out = []
    failed = 0
    for smi in smiles_list:
        try:
            out.append(sf.encoder(smi))
        except Exception:
            failed += 1
    return out, failed


def filter_selfies_len(selfies_list: list[str], max_len: int) -> list[str]:
    return [s for s in selfies_list if len(list(sf.split_selfies(s))) <= max_len]


tox21_train_smiles = load_smiles(TOX21_TRAIN_PATH)
tox21_val_smiles = load_smiles(TOX21_VAL_PATH)
tox21_test_smiles = load_smiles(TOX21_TEST_PATH)

tox21_train_selfies, tr_failed = smiles_to_selfies(tox21_train_smiles)
tox21_val_selfies, va_failed = smiles_to_selfies(tox21_val_smiles)
tox21_test_selfies, te_failed = smiles_to_selfies(tox21_test_smiles)

tox21_train_selfies = filter_selfies_len(tox21_train_selfies, max_len=MAX_LEN)
tox21_val_selfies = filter_selfies_len(tox21_val_selfies, max_len=MAX_LEN)
tox21_test_selfies = filter_selfies_len(tox21_test_selfies, max_len=MAX_LEN)

print(f"SELFIES conversion failures train/val/test: {tr_failed} / {va_failed} / {te_failed}")
print(f"tox21 filtered sizes train/val/test: {len(tox21_train_selfies):,} / {len(tox21_val_selfies):,} / {len(tox21_test_selfies):,}")


## Encode With Pretrained Tokenizer
This preserves compatibility with pretrained weights.
Any unseen token is mapped to `<UNK>`.


In [ ]:
def tokenize_selfies(s: str) -> list[str]:
    return list(sf.split_selfies(s))


def encode_selfies(s: str) -> list[int]:
    ids = [token_to_idx.get(tok, UNK_IDX) for tok in tokenize_selfies(s)]
    ids = ids[:MAX_LEN]
    ids.append(EOS_IDX)
    return ids


def encode_list(selfies_list: list[str]) -> np.ndarray:
    out = np.full((len(selfies_list), SEQ_LEN), PAD_IDX, dtype=np.int64)
    for i, s in enumerate(selfies_list):
        ids = encode_selfies(s)
        out[i, :len(ids)] = ids
    return out


def unk_rate(selfies_list: list[str]) -> float:
    total = 0
    unk = 0
    for s in selfies_list:
        for tok in tokenize_selfies(s)[:MAX_LEN]:
            total += 1
            if tok not in token_to_idx:
                unk += 1
    return 0.0 if total == 0 else unk / total

train_x = encode_list(tox21_train_selfies)
val_x = encode_list(tox21_val_selfies)
test_x = encode_list(tox21_test_selfies)

print("encoded shapes:", train_x.shape, val_x.shape, test_x.shape)
print(f"UNK rate train/val/test: {unk_rate(tox21_train_selfies):.4f} / {unk_rate(tox21_val_selfies):.4f} / {unk_rate(tox21_test_selfies):.4f}")


## Rebuild Model Class And Load Pretrained Weights
Architecture must match pretraining exactly for `load_state_dict` to succeed.


In [ ]:
class TokenDataset(Dataset):
    def __init__(self, x: np.ndarray):
        self.x = torch.from_numpy(x).long()

    def __len__(self):
        return self.x.size(0)

    def __getitem__(self, idx):
        return self.x[idx]


class PaperLikeSelfiesVAE(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, latent_dim: int = 292):
        super().__init__()
        self.vocab_size = vocab_size
        self.seq_len = seq_len

        self.conv_1 = nn.Conv1d(in_channels=seq_len, out_channels=9, kernel_size=9)
        self.conv_2 = nn.Conv1d(in_channels=9, out_channels=9, kernel_size=9)
        self.conv_3 = nn.Conv1d(in_channels=9, out_channels=10, kernel_size=11)
        self.relu = nn.ReLU()

        with torch.no_grad():
            dummy = torch.zeros(1, seq_len, vocab_size)
            d = self.relu(self.conv_1(dummy))
            d = self.relu(self.conv_2(d))
            d = self.relu(self.conv_3(d))
            flat = d.view(1, -1).size(1)

        self.linear_0 = nn.Linear(flat, 435)
        self.linear_1 = nn.Linear(435, latent_dim)
        self.linear_2 = nn.Linear(435, latent_dim)

        self.linear_3 = nn.Linear(latent_dim, 292)
        self.gru = nn.GRU(input_size=292, hidden_size=501, num_layers=3, batch_first=True)
        self.linear_4 = nn.Linear(501, vocab_size)

    def encoder(self, x_onehot: torch.Tensor):
        x = self.relu(self.conv_1(x_onehot))
        x = self.relu(self.conv_2(x))
        x = self.relu(self.conv_3(x))
        x = x.view(x.size(0), -1)
        x = F.selu(self.linear_0(x))
        return self.linear_1(x), self.linear_2(x)

    def sampling(self, mean: torch.Tensor, logvar: torch.Tensor):
        eps = 1e-2 * torch.randn_like(logvar)
        return torch.exp(0.5 * logvar) * eps + mean

    def decode(self, z: torch.Tensor):
        z = F.selu(self.linear_3(z))
        z_seq = z.unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.gru(z_seq)
        out2 = out.contiguous().view(-1, out.size(-1))
        probs = F.softmax(self.linear_4(out2), dim=1)
        return probs.contiguous().view(out.size(0), self.seq_len, self.vocab_size)

    def forward(self, x_onehot: torch.Tensor):
        mean, logvar = self.encoder(x_onehot)
        z = self.sampling(mean, logvar)
        probs = self.decode(z)
        return probs, mean, logvar


def ids_to_onehot(x_ids: torch.Tensor, vocab_size: int):
    return F.one_hot(x_ids, num_classes=vocab_size).float()


def vae_loss(pred_probs: torch.Tensor, x_onehot: torch.Tensor, mean: torch.Tensor, logvar: torch.Tensor):
    recon = F.binary_cross_entropy(pred_probs, x_onehot, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mean.pow(2) - logvar.exp())
    return recon + kl, recon, kl


model = PaperLikeSelfiesVAE(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, latent_dim=LATENT_DIM).to(device)
model.load_state_dict(ckpt["model_state_dict"], strict=True)

optim = torch.optim.Adam(model.parameters(), lr=POST_LR)
print("Loaded pretrained chembl_zinc weights successfully.")


## Optional: Resume From A Previously Post-Trained Checkpoint
Use this if a post-training run was interrupted and you want to continue from that exact point.
Leave `RESUME_POSTTRAIN = False` to start from pretrained chembl_zinc weights.


In [ ]:
RESUME_POSTTRAIN = False
RESUME_CKPT_PATH = POSTTRAIN_CKPT

if RESUME_POSTTRAIN:
    if not RESUME_CKPT_PATH.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {RESUME_CKPT_PATH}")

    resume_ckpt = torch.load(RESUME_CKPT_PATH, map_location="cpu")
    model.load_state_dict(resume_ckpt["model_state_dict"], strict=True)

    if "optimizer_state_dict" in resume_ckpt:
        optim.load_state_dict(resume_ckpt["optimizer_state_dict"])

    prev_hist = resume_ckpt.get("post_history", {})
    done_epochs = len(prev_hist.get("train_total", []))
    print(f"Resumed model/optimizer from {RESUME_CKPT_PATH}")
    print(f"Previously completed post-training epochs: {done_epochs}")
else:
    print("RESUME_POSTTRAIN is False. Starting from pretrained chembl_zinc weights.")


## Post-Training Helpers
Training below continues from pretrained weights.
Set `RUN_POST_TRAIN = True` in the config cell to begin.


In [ ]:
def make_loader(x: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(TokenDataset(x), batch_size=batch_size, shuffle=shuffle)


def evaluate(model: nn.Module, x: np.ndarray, batch_size: int):
    loader = make_loader(x, batch_size=batch_size, shuffle=False)
    model.eval()

    total_sum, recon_sum, kl_sum, n = 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for x_ids in tqdm(loader, desc="eval", leave=False):
            x_ids = x_ids.to(device)
            x_onehot = ids_to_onehot(x_ids, VOCAB_SIZE)

            pred_probs, mean, logvar = model(x_onehot)
            total, recon, kl = vae_loss(pred_probs, x_onehot, mean, logvar)

            b = x_ids.size(0)
            total_sum += total.item()
            recon_sum += recon.item()
            kl_sum += kl.item()
            n += b

    return {
        "total": total_sum / n,
        "recon": recon_sum / n,
        "kl": kl_sum / n,
    }


def post_train(
    model: nn.Module,
    optim: torch.optim.Optimizer,
    train_x: np.ndarray,
    val_x: np.ndarray,
    epochs: int,
    batch_size: int,
    eval_every: int = 1,
):
    train_loader = make_loader(train_x, batch_size=batch_size, shuffle=True)
    history = {"train_total": [], "val_total": []}

    for ep in tqdm(range(1, epochs + 1), desc="post-train epochs"):
        model.train()
        total_sum, n = 0.0, 0

        for x_ids in tqdm(train_loader, desc=f"post ep {ep:03d} train", leave=False):
            x_ids = x_ids.to(device)
            x_onehot = ids_to_onehot(x_ids, VOCAB_SIZE)

            optim.zero_grad()
            pred_probs, mean, logvar = model(x_onehot)
            total, _, _ = vae_loss(pred_probs, x_onehot, mean, logvar)
            total.backward()
            optim.step()

            b = x_ids.size(0)
            total_sum += total.item()
            n += b

        train_loss = total_sum / n
        history["train_total"].append(train_loss)

        if ep % eval_every == 0:
            val_loss = evaluate(model, val_x, batch_size=batch_size)["total"]
        else:
            val_loss = np.nan

        history["val_total"].append(val_loss)
        print(f"post-train ep {ep:03d} | train={train_loss:.4f} val={val_loss:.4f}")

    return model, history


## Run Post-Training (Optional)
Set `RUN_POST_TRAIN = True` in the config cell, then run this cell.


In [ ]:
if RUN_POST_TRAIN:
    print("Starting post-training from pretrained chembl_zinc checkpoint...")
    model, post_history = post_train(
        model=model,
        optim=optim,
        train_x=train_x,
        val_x=val_x,
        epochs=POST_EPOCHS,
        batch_size=BATCH_SIZE,
        eval_every=EVAL_EVERY,
    )

    post_test_metrics = evaluate(model, test_x, batch_size=BATCH_SIZE)
    print("Post-training test metrics:", post_test_metrics)
else:
    print("RUN_POST_TRAIN is False. Set RUN_POST_TRAIN=True in the config cell to begin training.")
    post_history = None
    post_test_metrics = None


## Save Post-Trained Checkpoint
This saves everything needed to continue later without restarting from pretraining.


In [ ]:
SAVE_POSTTRAINED = False

if SAVE_POSTTRAINED:
    if post_history is None:
        raise RuntimeError("No post-training history found. Run post-training first.")

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optim.state_dict(),
            "token_to_idx": token_to_idx,
            "seq_len": SEQ_LEN,
            "vocab_size": VOCAB_SIZE,
            "max_len": MAX_LEN,
            "latent_dim": LATENT_DIM,
            "post_history": post_history,
            "post_test_metrics": post_test_metrics,
            "source_pretrain_checkpoint": str(PRETRAIN_CKPT),
        },
        POSTTRAIN_CKPT,
    )
    print(f"Saved post-trained checkpoint: {POSTTRAIN_CKPT}")
else:
    print("SAVE_POSTTRAINED is False. Set SAVE_POSTTRAINED=True to write checkpoint.")


## Optional: Plot Post-Training Curves


In [ ]:
if post_history is not None:
    plt.figure(figsize=(6, 4))
    plt.plot(post_history["train_total"], label="train")
    plt.plot(post_history["val_total"], label="val")
    plt.xlabel("epoch")
    plt.ylabel("avg loss per sample")
    plt.title("Post-training on tox21")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No post-training history to plot yet.")
